In [1]:
import os
import joblib
import numpy as np
import pandas as pd
from pydantic import BaseModel
from typing import Optional


In [2]:

# --------- Models Loader ---------
def load_models(models_dir: str = 'models') -> Optional[dict]:
    try:
        return {
            'humidity_model': joblib.load(os.path.join(models_dir, 'humidity_model.pkl')),
            'crop_model': joblib.load(os.path.join(models_dir, 'crop_prediction_model.pkl')),
            'column_transformer': joblib.load(os.path.join(models_dir, 'column_transformer.pkl')),
            'scaler': joblib.load(os.path.join(models_dir, 'standard_scaler.pkl')),
            'label_encoder': joblib.load(os.path.join(models_dir, 'label_encoder.pkl')),
        }
    except FileNotFoundError as e:
        print(f"[ERROR] Loading models: {e}")
        return None


In [3]:
models = load_models()

In [4]:
models

{'humidity_model': RandomForestRegressor(random_state=42),
 'crop_model': BaggingClassifier(max_features=0.7, n_estimators=50, random_state=42),
 'column_transformer': ColumnTransformer(remainder='passthrough',
                   transformers=[('encoder', OneHotEncoder(),
                                  ['State_Name', 'Crop_Type'])]),
 'scaler': StandardScaler(with_mean=False),
 'label_encoder': LabelEncoder()}

In [ ]:

# --------- Input Schema ---------
class CropFeatures(BaseModel):
    N: float
    P: float
    K: float
    temperature: float
    rainfall: float
    ph: float
    State_Name: str
    Crop_Type: str
    Area_in_hectares: float
    Production_in_tons: float
    Yield_ton_per_hec: float
    Humidity_calculated: Optional[float] = None


# --------- Crop Prediction Logic ---------
def predict_crop(features: CropFeatures, models: dict) -> dict:
    features_dict = features.dict()
    features_df = pd.DataFrame([features_dict])

    if features.Humidity_calculated is None:
        humidity_features = features_df[['N', 'P', 'K', 'rainfall', 'temperature', 'ph']]
        humidity = models['humidity_model'].predict(humidity_features)[0]
        features_df['Humidity_calculated'] = humidity
    else:
        humidity = features.Humidity_calculated

    X = models['column_transformer'].transform(features_df)
    X = models['scaler'].transform(X)

    crop_id = models['crop_model'].predict(X)[0]
    crop_name = models['label_encoder'].inverse_transform([crop_id])[0]

    confidence = 0.0
    if hasattr(models['crop_model'], 'predict_proba'):
        probabilities = models['crop_model'].predict_proba(X)[0]
        confidence = probabilities[crop_id]

    return {
        "predicted_crop": crop_name,
        "confidence": float(confidence),
        "humidity_calculated": float(humidity)
    }


In [10]:
example_features = {
    'N': 90,
    'P': 42,
    'K': 43,
    'temperature': 20.87,
    'rainfall': 200.9,
    'ph': 6.5,
    'State_Name': 'gujarat',
    'Crop_Type': 'kharif',
    'Area_in_hectares': 2.0,
    'Production_in_tons': 4.0,
    'Yield_ton_per_hec': 2.0
}
f = CropFeatures(**example_features)
f

CropFeatures(N=90.0, P=42.0, K=43.0, temperature=20.87, rainfall=200.9, ph=6.5, State_Name='gujarat', Crop_Type='kharif', Area_in_hectares=2.0, Production_in_tons=4.0, Yield_ton_per_hec=2.0, Humidity_calculated=None)

In [11]:
predict_crop(f,models)

C:\Users\lenovo\AppData\Local\Temp\ipykernel_5172\1476096240.py:19: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  features_dict = features.dict()


{'predicted_crop': 'barley',
 'confidence': 0.7,
 'humidity_calculated': 82.22033820619994}